# Final coefficient-estimation experiment

Compare minimum width, OLS, Yule–Walker, and Horváth L-infinity with the **true AR order supplied**.
For each order $p=1,2,3,4,5$, draw **100 Jones coefficient vectors once** and reuse each vector
under all five innovation shapes. Every retained series has **T=1000** observations.
There are **500 coefficient vectors and 2,500 series**, with four estimates per series.

| Shape | Alpha | Beta |
|---|---:|---:|
| Uniform | 1 | 1 |
| Symmetric U-shaped | 0.5 | 0.5 |
| Symmetric peaked | 2 | 2 |
| Right-skewed | 1 | 2 |
| Left-skewed | 2 | 1 |

Use the project's centered, variance-normalized Beta family: population innovation mean zero and
variance $1/3$ in every condition. Support endpoints depend on the shape parameters; the uniform
condition has support $[-1,1]$. Draw independent innovation paths for each coefficient/shape pair.

The notebook first creates and verifies an immutable input archive. Estimation then reads those
saved arrays, uses identical data across methods, and checkpoints each completed series.
Rerunning loads the existing inputs and completed fits. Changing settings creates a separate run.

Sources: [Jones (1987), pp. 134–138](https://doi.org/10.2307/2347544);
[NIST, §1.3.6.6.17, density and moments](https://www.itl.nist.gov/div898/handbook/eda/section3/eda366h.htm);
[implemented Jones sampler](../ar_tools/functions.py), `jones_ar_coefficients`;
[innovation normalization](../ar_tools/innovations.py), `beta_innovation_bounds`.

In [ ]:
import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython import get_ipython
from IPython.display import Markdown, display

AR_CODE_DIR = next(
    candidate.resolve()
    for root in (Path.cwd(), *Path.cwd().parents)
    for candidate in (root / "Code" / "Code_AR_processes", root)
    if (candidate / "ar_tools/functions.py").is_file()
)
PROJECT_ROOT = AR_CODE_DIR.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Code.Code_AR_processes.ar_tools import final_coefficient_estimation as study
study = importlib.reload(study)
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")
pd.set_option("display.max_rows", 110)
pd.set_option("display.max_columns", 12)

## 1. Frozen design and estimation settings

The dataset settings and fit settings are separate. Adjusting the optimizer does not replace or
resimulate inputs. The three seed streams are independent of the earlier tuning experiment.
The fixed burn-in is 2,000 discarded observations; simulation starts from zero history.
The generating spectral radius is retained for every coefficient vector. This finite burn-in
is an explicit design choice, not an exact draw from the stationary initial-state distribution.

In [ ]:
DATA_CONFIGURATION = study.default_dataset_configuration()
DATA_CONFIGURATION.update(
    orders=[1, 2, 3, 4, 5],
    models_per_order=100,
    sample_size=1000,
    burnin=2000,
    coefficient_seed=2026102301,
    innovation_seed=2026102302,
)
FIT_CONFIGURATION = study.default_fit_configuration()
FIT_CONFIGURATION.update(
    scale=0.1,
    half_life=100.0,
    iterations=2000,
    patience=200,
    delta=1e-8,
    improvement_tolerance=0.0,
    optimizer_seed=2026102303,
)
N_JOBS = 4
BOOTSTRAP_COUNT = 2000
BOOTSTRAP_SEED = 2026102304
EXPERIMENT_NAME = "ar_coefficient_estimation_final"
INPUT_ROOT = AR_CODE_DIR / "inputs" / EXPERIMENT_NAME
CSV_ROOT = AR_CODE_DIR / "outputs" / "csv" / EXPERIMENT_NAME

expected_models = len(DATA_CONFIGURATION["orders"]) * DATA_CONFIGURATION["models_per_order"]
expected_series = expected_models * len(DATA_CONFIGURATION["beta_shapes"])
print(f"Design: {expected_models} coefficient vectors, {expected_series:,} series, T={DATA_CONFIGURATION['sample_size']}")
display(pd.DataFrame(DATA_CONFIGURATION["beta_shapes"]))

## 2. Sample, save, and verify the 2,500 series

Run this cell before fitting. Inputs are saved under `../inputs/ar_coefficient_estimation_final/<dataset-id>/`:

- `dataset.npz`: original float64 series and true coefficient bank; load with `allow_pickle=False`.
- `models.csv`: order, coefficient-vector ID, coefficient seed, coefficients, and true spectral radius.
- `series_metadata.csv`: links each series to its coefficient vector, Beta shape, innovation seed,
  normalized innovation support, retained length, burn-in, and series checksum.
- `manifest.json`: complete design, generator fingerprint, software versions, and file checksums.

The NPZ arrays are the authoritative numerical inputs. Coefficient vectors have zero padding;
only their first $p$ entries are used. The completed archive is published by a directory rename
after validation. Existing corrupt or incomplete inputs raise an error and are not silently regenerated.
[Storage and validation implementation](../ar_tools/final_coefficient_estimation.py).

In [ ]:
dataset = study.prepare_coefficient_dataset(INPUT_ROOT, DATA_CONFIGURATION)
assert dataset["series"].shape == (expected_series, DATA_CONFIGURATION["sample_size"])
assert len(dataset["models"]) == expected_models
coverage = dataset["metadata"].groupby(["shape_id", "alpha", "beta", "order"]).size().unstack("order")
assert coverage.eq(DATA_CONFIGURATION["models_per_order"]).all().all()
assert dataset["metadata"].groupby(["order", "model_id"])["shape_id"].nunique().eq(len(DATA_CONFIGURATION["beta_shapes"])).all()
print(f"Dataset ID: {dataset['dataset_id']}")
print(f"Verified saved inputs: {dataset['directory']}")
display(coverage.rename_axis(columns="AR order p"))
display(dataset["models"].groupby("order")["true_spectral_radius"].agg(["min", "median", "max"]))

## 3. Fit the four estimators to the saved series

**Minimum width:** construct newest-first $(p+1)$-coordinate windows, start at $e_1$, and use
$$\gamma_m = \frac{0.1}{L}\,2^{-m/100},$$
where $L$ is the exact Euclidean diameter of the empirical cloud. Retain every strict width
improvement. Stop at 2,000 evaluated states or after patience 200, with cumulative patience-reset
threshold $\delta=10^{-8}$. There are no stationarity-triggered restarts or truth-based choices.
A nonstationary estimate remains an estimate and contributes to the accuracy score.
[Tuned step rule](../ar_tools/tuning.py), `tune_geometric_half_life_schedules`;
[shared optimizer](../ar_tools/descent.py), `projected_subgradient_empirical_width`.

**OLS:** no-intercept regression on the same windows. **Yule–Walker:** centered empirical
covariances with denominator T. **Horváth:** the existing no-intercept L-infinity LP, with
full-data feasibility and duality checks. These are the repository's established comparator
implementations; Horváth does not fit a residual center.
[Estimator implementations](../ar_tools/classical_selectors.py), `fit_ols_known_order`,
`yule_walker_coefficients`, and `horvath_linf_coefficients`.

Four fits per saved series give **10,000 estimates** at the default design size.
Each series gets a checkpoint. Repeating the cell reuses completed checkpoints, including
explicitly recorded failures. Fit fingerprints include the data ID, settings, and estimator source.
Recorded elapsed times are diagnostic timings from parallel fitting, not a controlled speed study.

In [ ]:
run = study.run_coefficient_estimation(dataset, CSV_ROOT, FIT_CONFIGURATION, n_jobs=N_JOBS)
estimates = run["results"]
assert len(estimates) == expected_series * len(study.METHODS)
assert not estimates.duplicated(["sample_id", "method"]).any()
print(f"Fit ID: {run['fit_id']}")
print(f"Saved estimates: {run['directory'] / 'estimates.csv'}")
display(estimates.groupby(["method", "status"]).size().unstack(fill_value=0))
if estimates["status"].ne("ok").any():
    display(estimates.loc[estimates["status"].ne("ok"), ["sample_id", "order", "shape_id", "method", "error_message"]])

## 4. Accuracy and Monte Carlo uncertainty

For each fit, score
$$\operatorname{MAE}(\widehat\phi,\phi) = \frac1p\sum_{j=1}^{p}|\widehat\phi_j-\phi_j|.$$
The primary result is the mean of these per-fit errors over the 100 coefficient vectors,
separately for each order, innovation shape, and method. Also save the median, 90th percentile,
maximum, Monte Carlo standard error, nonstationary-estimate counts, and failed-fit counts.
All finite extreme errors remain in the summaries. A failed fit makes an unconditional score
unavailable; the successful-fit mean is separately labelled and is not substituted for it.
[Shared MAE definition](../ar_tools/coefficient_errors.py), `coefficient_mae`.

Use 2,000 bootstrap resamples of coefficient-vector IDs within each order. The same resample
indices are used across all five shapes and four methods, retaining the paired design.
For paired differences, negative values mean lower error for minimum width.
The 95% percentile intervals summarize simulation uncertainty under this design.
[Summary implementation](../ar_tools/final_coefficient_estimation.py), `summarize_coefficient_estimation`;
[Morris et al. (2019), §§3.2 and 5.2–5.3](https://doi.org/10.1002/sim.8086).

In [ ]:
accuracy, paired = study.summarize_coefficient_estimation(
    run, bootstrap_count=BOOTSTRAP_COUNT, bootstrap_seed=BOOTSTRAP_SEED,
)
mean_mae_table = accuracy.pivot(index=["alpha", "beta", "order"], columns="method", values="mean_mae").reindex(columns=study.METHODS)
display(mean_mae_table.style.format("{:.6g}"))
display(accuracy[["order", "shape_id", "method", "failed_fits", "nonstationary_fits", "mae_mcse"]])

## 5. Export the thesis experiment figures

The first figure shows mean and median coefficient MAE against order, separately for the five
Beta shapes. Mean curves have bootstrap intervals. The second shows paired mean MAE differences
between minimum width and each comparator. Every displayed figure is saved as both PDF and PNG
under the data/fit IDs in `../outputs/pdfs/` and `../outputs/images/`.

In [ ]:
PDF_DIR = AR_CODE_DIR / "outputs" / "pdfs" / EXPERIMENT_NAME / dataset["dataset_id"] / run["fit_id"]
IMAGE_DIR = AR_CODE_DIR / "outputs" / "images" / EXPERIMENT_NAME / dataset["dataset_id"] / run["fit_id"]
figure_paths = study.plot_coefficient_estimation(accuracy, paired, PDF_DIR, IMAGE_DIR)
for figure_path in figure_paths:
    print(figure_path)

In [ ]:
display(paired[["order", "shape_id", "comparator", "mean_mae_difference", "ci_lower", "ci_upper", "available_pairs"]])
print(f"Saved input archive: {dataset['directory']}")
print(f"Saved estimator outputs: {run['directory']}")
print("Main files: estimates.csv, accuracy_summary.csv, paired_comparisons.csv, failed_fits.csv")
print("Provenance: configuration.json, completion.json, summary_configuration.json")

## Scope of the experiment

This is a coefficient-estimation comparison with known order, T=1000, the five specified
innovation shapes, and the frozen optimizer configuration. Averaging is over Jones coefficient
draws and simulated innovation paths within each condition. The coefficient vectors are paired
across innovation shapes; each condition has one path per coefficient vector. This is not a
repeated-path performance estimate conditional on one fixed coefficient vector.

The minimum-width settings were selected in the earlier AR(3), uniform-innovation tuning study;
this experiment evaluates their use across orders and shapes without retuning on the evaluation
samples. Generating coefficients enter only the scoring and diagnostics after fitting.
[Earlier tuning experiment](../xx_notebooks_tuning/ar_tuning_experiments.ipynb).

This notebook is the sole estimation entry point for the final paired design. Historical
experiment data and figures remain saved. It writes no files under the protected thesis directory.